In [ ]:
import os
import pandas as pd
from bs4 import BeautifulSoup
import re

# 1. Đường dẫn tới thư mục chứa dữ liệu gốc
ROOT_DIR = "../data/html/03_articles"

# 2. Bộ từ khóa phân loại (ĐÂY LÀ ĐIỂM CHỐT ĐỂ LỌC GIÁ GẠO/TIÊU)
COFFEE_KEYWORDS = ["cà phê", "cafe","coffee" "robusta", "arabica"]

# Danh sách các tỉnh trọng điểm trồng cà phê
REGIONS = [
    "lâm đồng", "đắk lắk", "đắk nông", "gia lai", "kon tum", 
    "bảo lộc", "di linh", "cư m'gar", "buôn hồ", "buôn ma thuột"
]

# 3. Biểu thức Regex bắt giá tiền nội địa
vietnam_price_pattern = re.compile(
    r'\b(\d{2,3}[.,]\d{3})\s*(?:đồng|vnd|vnđ|đ)\s*/\s*kg\b', 
    re.IGNORECASE
)

data_rows = []

print("Đang quét và làm sạch dữ liệu từ thư mục...")

for root, dirs, files in os.walk(ROOT_DIR):
    for file in files:
        if file.endswith(".html"):
            filepath = os.path.join(root, file)
            url_hash = os.path.splitext(file)[0]
            publish_date = os.path.basename(root)
            
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    soup = BeautifulSoup(f.read(), "html.parser")
                    content = soup.get_text(separator=" ", strip=True)
            except Exception:
                continue
            
            # Cắt thành từng câu để thu hẹp phạm vi ngữ cảnh
            sentences = re.split(r'[\n|.!??]+', content)
            
            for sentence in sentences:
                sentence = sentence.strip()
                if not sentence:
                    continue
                
                sentence_lower = sentence.lower()
                
                # --- CỔNG LỌC NGỮ NGHĨA (CHỈ LẤY CÀ PHÊ) ---
                # Nếu câu không hề nhắc đến "cà phê", "robusta" hay "arabica" -> Bỏ qua ngay lập tức
                if not any(kw in sentence_lower for kw in COFFEE_KEYWORDS):
                    continue
                
                # Nếu qua được cổng lọc, mới bắt đầu tìm kiếm con số giá trị
                matches = vietnam_price_pattern.findall(sentence)
                
                if matches:
                    # Xác định vùng miền
                    found_regions = [r.title() for r in REGIONS if r in sentence_lower]
                    region_str = ", ".join(found_regions) if found_regions else "Toàn quốc"
                    
                    exact_prices = [f"{price_num} đồng/kg" for price_num in matches]
                    
                    data_rows.append({
                        "date": publish_date,
                        "url_hash": url_hash,
                        "region": region_str,
                        "exact_price": ", ".join(exact_prices),
                        "content_snippet": sentence
                    })

# 4. Lưu kết quả
df = pd.DataFrame(data_rows)


Đang quét và làm sạch dữ liệu từ thư mục...
✅ Hoàn tất! Đã trích xuất và lọc được 1185 mức giá (chỉ lấy Cà Phê).
📁 Kết quả sạch lưu tại: gia_ca_phe_noi_dia_sach.csv


In [ ]:
import pandas as pd

# 1. Ép kiểu cột 'date' về định dạng chuẩn Datetime của Pandas
# Điều này đảm bảo Pandas hiểu đây là thời gian thực sự chứ không phải văn bản
df['date'] = pd.to_datetime(df['date'])

# 2. Sắp xếp DataFrame theo thứ tự ngày tháng tăng dần (từ quá khứ đến hiện tại)
# inplace=True giúp cập nhật trực tiếp lên biến df hiện tại mà không cần gán lại
df.sort_values(by='date', ascending=True, inplace=True)

# 3. Reset lại Index cho gọn gàng sau khi các hàng bị xáo trộn vị trí
df.reset_index(drop=True, inplace=True)

# In thử 5 dòng đầu và 5 dòng cuối để kiểm tra kết quả
df.to_csv('gia_ca_phe_noi_dia.csv', index=False, encoding="utf-8-sig")

,date,url_hash,region,exact_price,content_snippet
0,2023-06-09,eeab3add0c957aa1f8a8401b74d00b7ed48d6a1a0cc14b...,Toàn quốc,"61,000 đồng/kg",Giá cà phê ngày 23/05/2023: Giá trong nước vượ...
1,2023-10-17,0d31cb3e65089fb36d7993536f973bd1b869fd87c01a42...,Toàn quốc,"113,900 đồng/kg","Tại Tây Nguyên, giá cà phê hiện là 113,300 - 1..."
2,2023-10-21,7ea1be22ced31ec219eb94e03acfcb48ac90a1c97d0d8f...,Đắk Nông,"57,000 đồng/kg","Tại tỉnh Đắk Nông, cà phê được thu mua với giá..."
3,2023-10-23,8a9ce0e2bab37eefbd138f4889804273d7b04d6adda191...,Đắk Nông,"60,100 đồng/kg","Tại tỉnh Đắk Nông, cà phê được thu mua với giá..."
4,2023-10-25,719723aa6423820f3eb0604acb1e9358e4616a13975166...,Đắk Nông,"61,100 đồng/kg","Tại tỉnh Đắk Nông, cà phê được thu mua với giá..."
...,...,...,...,...,...
1180,2025-12-15,b940a7f588d37976b9f92b3eaeda8a7dad28369241fb9d...,Đắk Nông,"95,500 đồng/kg",Tại vùng Đắk Nông cũ đang thu mua cà phê ở mức...
1181,2025-12-17,4bf26fd8a2750c06ac4a7dcff22829c57a33752514b7fd...,Đắk Lắk,"93,300 đồng/kg","Tương tự, giá cà phê tại tỉnh Đắk Lắk có mức g..."
1182,2025-12-17,4bf26fd8a2750c06ac4a7dcff22829c57a33752514b7fd...,Toàn quốc,"93,500 đồng/kg","com cập nhật chiều tối 17/12, giá cà phê tại c..."
1183,2025-12-17,4bf26fd8a2750c06ac4a7dcff22829c57a33752514b7fd...,Đắk Nông,"93,500 đồng/kg",Tại vùng Đắk Nông cũ đang thu mua cà phê ở mức...


In [36]:
df_price=pd.read_csv('gia_ca_phe_noi_dia.csv', encoding="utf-8-sig")
df_info=pd.read_csv('data/html/coffee_articles.csv', encoding="utf-8-sig")

In [38]:
df_info

,URL_HASH,TARGET,QUERY_ID,DOMAIN,URL,PUBLISHED_DATE,DISCOVERED_AT,CONTENT
0,0378154854331b7a33930dcdbe5464e49fcec7aa1e11ed...,robusta,robusta_general,www.konnaicoffee.vn,https://www.konnaicoffee.vn/konnai-coffee-huon...,2024-06-22,2026-03-19 18:29:49.333,Konnai Coffee: Hương vị Sơn La vang danh châu ...
1,4b69332104762862f48aa41bb6790772e1f7060a09e105...,robusta,robusta_general,thoibaotaichinhvietnam.vn,https://thoibaotaichinhvietnam.vn/gia-vang-hom...,2024-06-22,2026-03-19 18:29:48.327,"Giá vàng hôm nay (22/6): Vàng thế giới ""quay x..."
2,2c1a0eb25e8f183257484a1c9de97b532d58f07cbb735e...,robusta,robusta_general,baoquocte.vn,https://baoquocte.vn/gia-tieu-hom-nay-2262024-...,2024-06-22,2026-03-19 18:29:47.347,"Giá tiêu hôm nay 22/6/2024, xuất khẩu dự kiến ..."
3,fe316f2899875778af727f5b62fc2cf6dc9b098a9ee506...,robusta,robusta_local,nongnghiephuuco.vn,https://nongnghiephuuco.vn/hieu-thi-truong-hop...,2024-06-22,2026-03-19 18:29:44.289,"Hiểu thị trường, hợp tác và xây dựng thương hi..."
4,29bf8775272dc1ec1beb8f62283e74e390851e42850d9b...,robusta,robusta_local,thaichaucoffee.com,https://thaichaucoffee.com/tin-tuc/ca-phe-thai...,2024-06-22,2026-03-19 18:29:42.762,CÀ PHÊ THÁI CHÂU ĐẶC BIỆT\n\nhttps://analytics...
...,...,...,...,...,...,...,...,...
9548,5ae78e703e7fe671c340ec41ea34c10df15e020d4db969...,arabica,arabica_explicit,carocoffee.com,https://carocoffee.com/goc-ca-phe,2025-04-26,2026-03-19 20:29:08.568,Caro Coffee - GÓC CÀ PHÊ\n\nĐơn hàng\nĐăng nhậ...
9549,76aebd53a441bec73c1fee6566c60b19b7fc3ae3fffc74...,robusta,robusta_local,baonghean.vn,https://baonghean.vn/gia-ca-phe-hom-nay-26-4-2...,2025-04-26,2026-03-19 20:29:09.825,Giá cà phê hôm nay 26/4/2025: Tiếp tục tăng ...
9550,8491c07be56a258ad01ac5d85dc3fd3e0c8967cd552407...,robusta,robusta_local,baodanang.vn,https://baodanang.vn/gia-vang-hom-nay-29-4-202...,2025-04-29,2026-03-19 20:30:45.474,Giá vàng hôm nay 29/4/2025: Giá vàng chìm tron...
9551,182fb02b17f057cd384fe8f0bd845e9506dee4c54a6b8b...,robusta,robusta_local,trangtraiviet.danviet.vn,https://trangtraiviet.danviet.vn/gia-ca-phe-li...,2025-05-03,2026-03-19 20:31:47.451,"Giá cà phê liên tục đảo chiều, lại tăng lên mạ..."


In [39]:
df_price

,date,url_hash,region,exact_price,content_snippet
0,2023-06-09,eeab3add0c957aa1f8a8401b74d00b7ed48d6a1a0cc14b...,Toàn quốc,"61,000 đồng/kg",Giá cà phê ngày 23/05/2023: Giá trong nước vượ...
1,2023-10-17,0d31cb3e65089fb36d7993536f973bd1b869fd87c01a42...,Toàn quốc,"113,900 đồng/kg","Tại Tây Nguyên, giá cà phê hiện là 113,300 - 1..."
2,2023-10-21,7ea1be22ced31ec219eb94e03acfcb48ac90a1c97d0d8f...,Đắk Nông,"57,000 đồng/kg","Tại tỉnh Đắk Nông, cà phê được thu mua với giá..."
3,2023-10-23,8a9ce0e2bab37eefbd138f4889804273d7b04d6adda191...,Đắk Nông,"60,100 đồng/kg","Tại tỉnh Đắk Nông, cà phê được thu mua với giá..."
4,2023-10-25,719723aa6423820f3eb0604acb1e9358e4616a13975166...,Đắk Nông,"61,100 đồng/kg","Tại tỉnh Đắk Nông, cà phê được thu mua với giá..."
...,...,...,...,...,...
1180,2025-12-15,b940a7f588d37976b9f92b3eaeda8a7dad28369241fb9d...,Đắk Nông,"95,500 đồng/kg",Tại vùng Đắk Nông cũ đang thu mua cà phê ở mức...
1181,2025-12-17,4bf26fd8a2750c06ac4a7dcff22829c57a33752514b7fd...,Đắk Lắk,"93,300 đồng/kg","Tương tự, giá cà phê tại tỉnh Đắk Lắk có mức g..."
1182,2025-12-17,4bf26fd8a2750c06ac4a7dcff22829c57a33752514b7fd...,Toàn quốc,"93,500 đồng/kg","com cập nhật chiều tối 17/12, giá cà phê tại c..."
1183,2025-12-17,4bf26fd8a2750c06ac4a7dcff22829c57a33752514b7fd...,Đắk Nông,"93,500 đồng/kg",Tại vùng Đắk Nông cũ đang thu mua cà phê ở mức...


In [40]:


# Giả sử bạn đã đọc dữ liệu từ CSV vào 2 biến DataFrame
# df_info = pd.read_csv('đường_dẫn_file_info.csv')
# df_price = pd.read_csv('đường_dẫn_file_price.csv')

# 1. Chuẩn hóa tên cột của df_info về chữ thường (Best Practice trong Data Cleaning)
df_info.columns = df_info.columns.str.lower()

# 2. Lọc các cột cần thiết từ bảng Info để tránh trùng lặp dữ liệu (ví dụ trùng cột date)
# Ở đây ta lấy url_hash làm khóa, kéo thêm target (Arabica/Robusta), domain (Nguồn tin) và url (Link gốc)
info_columns_to_keep = ['url_hash', 'target', 'domain', 'url']
df_info_subset = df_info[info_columns_to_keep]

# 3. Thực hiện Merge
# Dùng 'left' join với df_price bên trái để giữ nguyên vẹn 1185 dòng giá trị đã bóc tách
df_merged = pd.merge(
    df_price, 
    df_info_subset, 
    on='url_hash', 
    how='left'
)

# 4. Kiểm tra chất lượng dữ liệu sau khi merge (Data Quality Check)
print(f"Số dòng trước khi merge: {len(df_price)}")
print(f"Số dòng sau khi merge: {len(df_merged)}")

# Kiểm tra xem có dòng nào không map được thông tin không (Null check)
missing_info = df_merged['domain'].isnull().sum()
if missing_info > 0:
    print(f"Cảnh báo: Có {missing_info} dòng giá không tìm thấy thông tin URL tương ứng trong df_info.")

# 5. Lưu thành Data Warehouse dạng phẳng (Flat table) sẵn sàng cho phân tích
output_path = "coffee_master_dataset.csv"
df_merged.to_csv(output_path, index=False, encoding="utf-8-sig")

df_merged

Số dòng trước khi merge: 1185
Số dòng sau khi merge: 1185


,date,url_hash,region,exact_price,content_snippet,target,domain,url
0,2023-06-09,eeab3add0c957aa1f8a8401b74d00b7ed48d6a1a0cc14b...,Toàn quốc,"61,000 đồng/kg",Giá cà phê ngày 23/05/2023: Giá trong nước vượ...,robusta,caphehat.vn,https://caphehat.vn/tin-tuc-1/?srsltid=AfmBOor...
1,2023-10-17,0d31cb3e65089fb36d7993536f973bd1b869fd87c01a42...,Toàn quốc,"113,900 đồng/kg","Tại Tây Nguyên, giá cà phê hiện là 113,300 - 1...",robusta,tapchicongthuong.vn,https://tapchicongthuong.vn/gia-ca-phe-hom-nay...
2,2023-10-21,7ea1be22ced31ec219eb94e03acfcb48ac90a1c97d0d8f...,Đắk Nông,"57,000 đồng/kg","Tại tỉnh Đắk Nông, cà phê được thu mua với giá...",robusta,vov.vn,https://vov.vn/thi-truong/gia-ca-phe/gia-ca-ph...
3,2023-10-23,8a9ce0e2bab37eefbd138f4889804273d7b04d6adda191...,Đắk Nông,"60,100 đồng/kg","Tại tỉnh Đắk Nông, cà phê được thu mua với giá...",robusta,congthuong.vn,https://congthuong.vn/gia-ca-phe-hom-nay-ngay-...
4,2023-10-25,719723aa6423820f3eb0604acb1e9358e4616a13975166...,Đắk Nông,"61,100 đồng/kg","Tại tỉnh Đắk Nông, cà phê được thu mua với giá...",robusta,kinhte.congthuong.vn,https://kinhte.congthuong.vn/gia-ca-phe-moi-nh...
...,...,...,...,...,...,...,...,...
1180,2025-12-15,b940a7f588d37976b9f92b3eaeda8a7dad28369241fb9d...,Đắk Nông,"95,500 đồng/kg",Tại vùng Đắk Nông cũ đang thu mua cà phê ở mức...,robusta,vov.vn,https://vov.vn/thi-truong/gia-ca-phe/gia-ca-ph...
1181,2025-12-17,4bf26fd8a2750c06ac4a7dcff22829c57a33752514b7fd...,Đắk Lắk,"93,300 đồng/kg","Tương tự, giá cà phê tại tỉnh Đắk Lắk có mức g...",robusta,vov.vn,https://vov.vn/thi-truong/gia-ca-phe/gia-ca-ph...
1182,2025-12-17,4bf26fd8a2750c06ac4a7dcff22829c57a33752514b7fd...,Toàn quốc,"93,500 đồng/kg","com cập nhật chiều tối 17/12, giá cà phê tại c...",robusta,vov.vn,https://vov.vn/thi-truong/gia-ca-phe/gia-ca-ph...
1183,2025-12-17,4bf26fd8a2750c06ac4a7dcff22829c57a33752514b7fd...,Đắk Nông,"93,500 đồng/kg",Tại vùng Đắk Nông cũ đang thu mua cà phê ở mức...,robusta,vov.vn,https://vov.vn/thi-truong/gia-ca-phe/gia-ca-ph...
